# Airport Investment Intelligence Agent — Notebook Demo

Each cell below runs the *exact* real CLI command as a subprocess — not a reimplementation, not a direct SDK call. This sidesteps Jupyter-specific quirks entirely (its own event loop, its own import path) by just doing what a terminal would do: start a clean process, run the command, print what came back.

In [ ]:
import subprocess
import sys
from pathlib import Path


def _find_repo_root(marker: str = "pyproject.toml") -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find repo root (no {marker} found above {current})")


PROJECT_ROOT = str(_find_repo_root())


def ask(question: str, session_id: str = "notebook-demo") -> None:
    result = subprocess.run(
        [sys.executable, "-m", "airport_investment_agent", question, "--session-id", session_id],
        capture_output=True,
        text=True,
        cwd=PROJECT_ROOT,
    )
    print(result.stdout.strip())
    if result.returncode != 0:
        print("--- stderr ---")
        print(result.stderr.strip())

## 1. Single-airport traffic analysis

In [3]:
ask("What is the current traffic situation at JFK?")

JFK traffic:
- **Passengers:** 31,011,429
- **Departures:** 220,083
- **Seats:** 37,884,404
- **Load factor:** 0.8186
- **Passengers per departure:** 140.91
- **Growth rate:** -0.0152
- **Completion rate:** 0.9892

Caveats:
- **BTS T-100 data, year 2025**
- Growth compares **2025 vs 2024**
- Completion rate from **BTS On-Time Performance data, 2026-05**


## 2. Comparing 2-5 airports for investment potential

In [ ]:
ask("Compare JFK, LAX, and ORD for investment potential.")

Ranking:
1. **ORD** — **0.9984**
2. **JFK** — **0.7104**
3. **LAX** — **0.7064**

Why:
- **ORD** has the strongest **volume** and **growth**.
- **JFK** slightly beats **LAX** overall thanks to better **efficiency** and growth.
- **LAX** has high volume but weaker growth.

Traffic:
- **ORD:** 40,849,250 passengers; load factor 0.8145; growth 0.0545
- **JFK:** 31,011,429 passengers; load factor 0.8186; growth -0.0152
- **LAX:** 36,766,887 passengers; load factor 0.8158; growth -0.0385

Caveats:
- **BTS T-100 data, year 2025**
- Growth compares **2025 vs 2024**


## 3. Unmet-demand / capacity strain explanation

In [ ]:
ask("Is there unmet demand at ATL? Explain your assumptions.")

Yes — ATL shows strong unmet-demand pressure.

- **Demand pressure proxy:** **0.98**
- **Volume:** **1.00**
- **Efficiency:** **0.96**

Assumptions:
- This is a **proxy**, not a direct count of unmet demand.
- It blends **traffic volume** and **load factor/efficiency**.
- High scores on both suggest the airport may be approaching capacity.

Caveats:
- No public data directly counts passengers who wanted to fly but couldn’t get a seat.
- Reference point is the busiest real U.S. commercial airport in the latest complete year.
- **BTS T-100 data, year 2025**
- Growth rate compares **2025 vs 2024**


## 4. News context (optional tool — context only, never a scoring input)

In [ ]:
ask("Any recent news about ORD that could affect investment there?")

Yes — recent ORD news looks **generally positive** for investment.

Highlights:
- Chicago broke ground on the **new $1.3 billion Concourse D** in **August 2025**
- That’s a major **ORDNext** terminal-expansion milestone
- ORD has also been highlighting **strong airline growth** and **record passenger volumes** in 2025–2026
- New carrier activity includes **Arajet’s ORD–Punta Cana route**
- **Air France** opened a new lounge at **Terminal 5**

Takeaway:
- **Positive long-term capital signal**
- Ongoing construction means **near-term operational disruption risk**

Caveat:
- This is **news context only**, not a scoring input.


## 5. Multi-turn session memory

Both calls below share the same explicit `session_id` (the CLI's own `--session-id` flag, built for exactly this) — the second question deliberately omits the airport codes already mentioned, relying on the shared session to carry that context.

In [ ]:
ask("Compare JFK and LAX.", session_id="memory-demo")
print("---")
ask("What about ORD too?", session_id="memory-demo")

JFK ranks above LAX.

- 1) JFK — investment_score 0.9335
  - passengers: 31,011,429
  - departures: 220,083
  - seats: 37,884,404
  - load_factor: 0.8185803582920296
  - growth_rate: -0.015204434900740612
  - caveats: BTS T-100 data, year 2025; growth rate compares 2025 vs 2024

- 2) LAX — investment_score 0.7489
  - passengers: 36,766,887
  - departures: 273,905
  - seats: 45,067,289
  - load_factor: 0.8158220255937738
  - growth_rate: -0.038545407589539345
  - caveats: BTS T-100 data, year 2025; growth rate compares 2025 vs 2024

LAX has higher volume, but JFK scores better overall.
---
With ORD included, the ranking is:

1) ORD — investment_score 0.9984  
2) JFK — investment_score 0.7104  
3) LAX — investment_score 0.7064  

Caveats: all three use BTS T-100 data, year 2025; growth rate compares 2025 vs 2024.
